# Act 2 — Tools and the ReAct Loop

"ReAct" = **Re**ason + **Act**: the model thinks about what to do, calls a tool, observes the result, and repeats until it has a final answer.

In [ ]:
import sys, os, json, inspect
sys.path.insert(0, "..")
from dotenv import load_dotenv
load_dotenv(override=True)

from litellm import acompletion
from mini_agent import Agent, calculator, web_search
from mini_agent.agent import tool_to_schema

MODEL = os.environ.get("MODEL", "deepseek/deepseek-v4-flash")

## A tool is just a Python function
No decorators, no special base class needed — type hints + a docstring are enough. Here's the simplest possible one:

In [ ]:
def add(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

Python already knows everything about it — its docstring and its signature:

In [ ]:
print(inspect.getdoc(add))
print(inspect.signature(add))

## Under the hood: `tool_to_schema`
Before calling anything, here's what a tool-calling API request actually needs — built from the exact signature and docstring you just inspected, in the same JSON-schema shape as the book's `function_to_input_schema`, condensed.

In [ ]:
print(json.dumps(tool_to_schema(add), indent=2))

## The raw way: one call to `acompletion`
Send that schema straight to the model with `litellm.acompletion`, and see whether it asks to call `add` instead of answering directly:

In [ ]:
messages = [{"role": "user", "content": "What is 7 + 5?"}]

response = await acompletion(
    model=MODEL,
    messages=messages,
    tools=[tool_to_schema(add)],
    tool_choice="auto",
)
message = response.choices[0].message
print("tool_calls:", message.tool_calls)

It asked for `add` instead of answering directly. Run it, and hand the result back:

In [ ]:
call = message.tool_calls[0]
args = json.loads(call.function.arguments)
result = add(**args)
print("result:", result)

messages.append(message.model_dump())
messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})

final = await acompletion(model=MODEL, messages=messages, tools=[tool_to_schema(add)])
print(final.choices[0].message.content)

## The same thing, organized: `Agent`
That was one schema, one manual tool-call check, one manual execution, one manual message append, one more API call — a lot of bookkeeping for a single round trip. `Agent` packages exactly that:

In [ ]:
agent = Agent(tools=[add], instructions="Use tools when they help.")
answer = await agent.run("What is 7 + 5?")
print("\nFINAL:", answer)

Same round trip, same schema, same tool execution — just organized into a class instead of copy-pasted every time you need it.

That's a full round trip — but only one. A harder question might need several tool calls in sequence before the model is ready to answer.

## Step 2 — wrap it in a loop
Same steps, repeated until the model stops asking for tools:

In [ ]:
async def run_with_tools(prompt, tools, max_steps=5):
    schemas = [tool_to_schema(t) for t in tools]
    by_name = {t.__name__: t for t in tools}
    messages = [{"role": "user", "content": prompt}]

    for step in range(max_steps):
        response = await acompletion(model=MODEL, messages=messages, tools=schemas, tool_choice="auto")
        message = response.choices[0].message
        messages.append(message.model_dump())

        if not message.tool_calls:
            return message.content

        for call in message.tool_calls:
            args = json.loads(call.function.arguments)
            result = by_name[call.function.name](**args)
            print(f"[step {step}] called {call.function.name}({args}) -> {result}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})

    return "(stopped: max_steps reached)"

print(await run_with_tools("What is (7 + 5) + (3 + 4)? Use the add tool for each addition, one at a time.", tools=[add]))

Two separate calls to `add`, chained by the loop, before it had enough to answer — that's think → act → observe, running for more than one step.

## Step 3 — that loop *is* `Agent`
Open `mini_agent/agent.py` and it's the same shape, organized into methods so it can be reused across tool sets and conversations instead of copy-pasted:

| What we just wrote | Becomes |
|---|---|
| `tools` → `schemas` + `by_name` | `Agent.__init__` |
| the `for step in range(max_steps)` loop | `Agent.run` |
| the single `acompletion` call inside it | `Agent._complete_with_retry` (same idea, plus retry-on-rate-limit) |
| the `for call in message.tool_calls` block | `Agent._invoke` |

Everything from here uses the real `Agent` — same mechanics, less bookkeeping on screen.

The tool we'll use for the rest of this notebook, `calculator`, is built the same way — just a bit more logic inside:

In [ ]:
print(inspect.getsource(calculator))

Same pattern, applied to a slightly more useful problem:

In [ ]:
agent = Agent(tools=[calculator], instructions="You are a helpful assistant. Use tools for math.")
answer = await agent.run("What is 8734 * 291, minus 100?")
print("\nFINAL:", answer)

## Different tool, same pattern
`web_search` needs a `TAVILY_API_KEY` in `.env` — without one, it tells you so instead of crashing the loop.

In [ ]:
agent2 = Agent(tools=[calculator, web_search], instructions="Use tools when they help; search for anything current.")
answer2 = await agent2.run("Search the web: who won the most recent Nobel Prize in Physics?")
print("\nFINAL:", answer2)

The agent decided *on its own* which tool to use, or whether to use one at all — driven entirely by the model reading each tool's schema. That's the whole trick behind "agentic" tool use.

**Next:** give it memory across turns, and let it write and run its own code → `03_memory_and_code_execution.ipynb`.